In [ ]:
import numpy as np
from ase.build import bulk
from mace.calculators import MACECalculator
import pandas as pd
from ase.optimize import BFGS
from ase.filters import UnitCellFilter
from oncapintada.subregular_model import BinaryAlloy


In [2]:
# ff = MACE-matpes-pbe-omat-ft.model | MACE-matpes-r2scan-omat-ft.model | mace-omat-0-medium.model
ff = 'MACE-matpes-pbe-omat-ft.model'
calc = MACECalculator(model_paths=f"/Users/leseixas/.local/mace/{ff}",
                      device='cpu',
                      default_dtype='float32') 

Default dtype float32 does not match model dtype float64, converting models to float32.


/Users/leseixas/miniconda3/envs/mace/lib/python3.13/site-packages/mace/calculators/mace.py:139: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


In [6]:
elements = ["Au", "Pt"]
energy_matrix = np.zeros((len(elements), len(elements)))
for j, element in enumerate(elements):
    atoms = bulk(element, cubic=True)
    atoms.calc = calc
    opt = BFGS(UnitCellFilter(atoms), logfile=None)
    opt.run(fmax=0.01)
    energy = atoms.get_potential_energy()
    print(f"{element}: {energy:.6f} eV")

    # SUPERCELL
    supercell = atoms.repeat((2, 2, 2)).copy()
    supercell.calc = calc
    opt_supercell = BFGS(UnitCellFilter(supercell), logfile=None)
    opt_supercell.run(fmax=0.01)
    energy_supercell = supercell.get_potential_energy()
    energy_matrix[j, j] = energy_supercell
    print(f"{element} supercell: {energy_supercell:.6f} eV")

    # IMPURITIES
    for i, impurity in enumerate(elements):
        if impurity != element:
            impurity_atoms = supercell.copy()
            impurity_atoms[0].symbol = impurity
            impurity_atoms.calc = calc
            opt_impurity = BFGS(UnitCellFilter(impurity_atoms), logfile=None)
            opt_impurity.run(fmax=0.01)
            energy_impurity = impurity_atoms.get_potential_energy()
            print(f"{element} with {impurity} impurity: {energy_impurity:.6f} eV")
            energy_matrix[i, j] = energy_impurity



Au: -12.895026 eV
Au supercell: -103.160202 eV
Au with Pt impurity: -105.836197 eV
Pt: -24.377337 eV
Pt supercell: -195.018661 eV
Pt with Au impurity: -191.818298 eV


In [7]:
energy_matrix

array([[-103.16020203, -191.81829834],
       [-105.8361969 , -195.0186615 ]])